In [2]:
!nvcc --version || true

import numba
print("numba:", numba.__version__)

from numba import cuda
cuda.cudadrv.libs.test()

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
numba: 0.60.0
Finding driver from candidates:
	libcuda.so
	libcuda.so.1
	/usr/lib/libcuda.so
	/usr/lib/libcuda.so.1
	/usr/lib64/libcuda.so
	/usr/lib64/libcuda.so.1
Using loader <class 'ctypes.CDLL'>
	Trying to load driver...	ok
		Loaded from libcuda.so
	Mapped libcuda.so paths:
		/usr/lib64-nvidia/libcuda.so.580.82.07
Finding nvvm from NVIDIA NVCC Wheel
	Located at /usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/nvvm/lib64/libnvvm.so
	Trying to open library...	ok
Finding nvrtc from NVIDIA NVCC Wheel
	Located at /usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12
	Trying to open library...	ok
Finding cudadevrt from System
	Located at /usr/local/cuda/lib64/libcudadevrt.a
	Checking library...	ok
Finding libdevice from NVIDIA NVCC Wheel
	Located at /us

True

In [1]:
import os
os.environ["NUMBA_FORCE_CUDA_CC"] = "9.0"

In [ ]:
import hashlib
import time

import numpy as np
from numba import cuda, uint8, uint32, uint64, int32


K_HOST = np.array([
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
], dtype=np.uint32)


MAX_THREADS_PER_BLOCK = 1024
UINT64_MAX = (1 << 64) - 1


@cuda.jit(device=True)
def rotr(x, n):
    return uint32((x >> n) | (x << (32 - n)))


@cuda.jit(device=True)
def sha256_one_block(block, digest, K):
    w = cuda.local.array(64, dtype=uint32)

    for i in range(16):
        j = i * 4
        w[i] = uint32(
            (uint32(block[j]) << 24)
            | (uint32(block[j + 1]) << 16)
            | (uint32(block[j + 2]) << 8)
            | uint32(block[j + 3])
        )

    for i in range(16, 64):
        s0 = rotr(w[i - 15], 7) ^ rotr(w[i - 15], 18) ^ (w[i - 15] >> 3)
        s1 = rotr(w[i - 2], 17) ^ rotr(w[i - 2], 19) ^ (w[i - 2] >> 10)
        w[i] = uint32(w[i - 16] + s0 + w[i - 7] + s1)

    a = uint32(0x6a09e667)
    b = uint32(0xbb67ae85)
    c = uint32(0x3c6ef372)
    d = uint32(0xa54ff53a)
    e = uint32(0x510e527f)
    f = uint32(0x9b05688c)
    g = uint32(0x1f83d9ab)
    h = uint32(0x5be0cd19)

    for i in range(64):
        S1 = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25)
        ch = (e & f) ^ ((e ^ uint32(0xffffffff)) & g)
        temp1 = uint32(h + S1 + ch + K[i] + w[i])

        S0 = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22)
        maj = (a & b) ^ (a & c) ^ (b & c)
        temp2 = uint32(S0 + maj)

        h = g
        g = f
        f = e
        e = uint32(d + temp1)
        d = c
        c = b
        b = a
        a = uint32(temp1 + temp2)

    digest[0] = uint32(a + uint32(0x6a09e667))
    digest[1] = uint32(b + uint32(0xbb67ae85))
    digest[2] = uint32(c + uint32(0x3c6ef372))
    digest[3] = uint32(d + uint32(0xa54ff53a))
    digest[4] = uint32(e + uint32(0x510e527f))
    digest[5] = uint32(f + uint32(0x9b05688c))
    digest[6] = uint32(g + uint32(0x1f83d9ab))
    digest[7] = uint32(h + uint32(0x5be0cd19))


@cuda.jit(device=True)
def count_leading_f_nibbles(digest):
    count = 0

    for i in range(8):
        word = digest[i]

        for j in range(8):
            shift = 28 - j * 4
            nibble = (word >> shift) & uint32(0xF)

            if nibble == uint32(0xF):
                count += 1
            else:
                return count

    return count


@cuda.jit
def mine_max_f_kernel(
    prefix,
    prefix_len,
    start_nonce,
    block_best_scores,
    block_best_nonces,
    K,
):
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    idx = cuda.grid(1)

    nonce = uint64(start_nonce) + uint64(idx)

    msg_block = cuda.local.array(64, dtype=uint8)
    digest = cuda.local.array(8, dtype=uint32)

    s_scores = cuda.shared.array(MAX_THREADS_PER_BLOCK, dtype=int32)
    s_nonces = cuda.shared.array(MAX_THREADS_PER_BLOCK, dtype=uint64)

    for i in range(64):
        msg_block[i] = uint8(0)

    for i in range(prefix_len):
        msg_block[i] = prefix[i]

    pos = prefix_len

    # prefix + nonce
    # nonceは8バイト big-endian
    for i in range(8):
        shift = (7 - i) * 8
        msg_block[pos + i] = uint8((nonce >> shift) & uint64(0xff))

    msg_len = prefix_len + 8

    # SHA-256 padding
    msg_block[msg_len] = uint8(0x80)

    bit_len = uint64(msg_len * 8)

    for i in range(8):
        shift = (7 - i) * 8
        msg_block[56 + i] = uint8((bit_len >> shift) & uint64(0xff))

    sha256_one_block(msg_block, digest, K)

    score = count_leading_f_nibbles(digest)

    s_scores[tid] = score
    s_nonces[tid] = nonce

    cuda.syncthreads()

    # block内で最大scoreを探す
    stride = cuda.blockDim.x // 2

    while stride > 0:
        if tid < stride:
            other_score = s_scores[tid + stride]
            other_nonce = s_nonces[tid + stride]

            my_score = s_scores[tid]
            my_nonce = s_nonces[tid]

            if other_score > my_score or (
                other_score == my_score and other_nonce < my_nonce
            ):
                s_scores[tid] = other_score
                s_nonces[tid] = other_nonce

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        block_best_scores[bid] = s_scores[0]
        block_best_nonces[bid] = s_nonces[0]


def sha256_hex(prefix_bytes: bytes, nonce: int) -> str:
    msg = prefix_bytes + nonce.to_bytes(8, "big")
    return hashlib.sha256(msg).hexdigest()


def is_power_of_two(x: int) -> bool:
    return x > 0 and (x & (x - 1)) == 0


def device_name() -> str:
    name = cuda.get_current_device().name
    if isinstance(name, bytes):
        return name.decode()
    return str(name)


def run(
    prefix="mumumu-",
    start=0,
    threads=256,
    blocks=8192,
    device=0,
    progress_sec=1.0,
    max_batches=None,
):
    if threads > MAX_THREADS_PER_BLOCK:
        raise ValueError(f"threads は {MAX_THREADS_PER_BLOCK} 以下にしてね")

    if not is_power_of_two(threads):
        raise ValueError("threads は 2の累乗にしてね。例: 128, 256, 512, 1024")

    if start < 0 or start > UINT64_MAX:
        raise ValueError("start は 0 以上 2^64-1 以下にしてね")

    prefix_bytes = prefix.encode("utf-8")

    # この実装は1ブロックSHA-256専用
    # prefix + 8byte nonce が55バイト以下である必要がある
    if len(prefix_bytes) + 8 > 55:
        raise ValueError("prefix は47バイト以下にしてね")

    cuda.select_device(device)

    print("device:", device_name())
    print("prefix:", prefix_bytes)
    print("threads:", threads)
    print("blocks:", blocks)
    print("batch size:", threads * blocks)
    print()

    prefix_np = np.frombuffer(prefix_bytes, dtype=np.uint8).copy()

    d_prefix = cuda.to_device(prefix_np)
    d_K = cuda.to_device(K_HOST)

    d_block_best_scores = cuda.device_array(blocks, dtype=np.int32)
    d_block_best_nonces = cuda.device_array(blocks, dtype=np.uint64)

    print("compiling kernel...")
    mine_max_f_kernel[1, threads](
        d_prefix,
        len(prefix_bytes),
        np.uint64(0),
        d_block_best_scores,
        d_block_best_nonces,
        d_K,
    )
    cuda.synchronize()

    print("start")
    print()

    batch_size = threads * blocks
    next_nonce = start
    checked = 0
    batches = 0

    global_best_score = -1
    global_best_nonce = 0
    global_best_hash = ""

    t0 = time.perf_counter()
    last_progress = t0

    while True:
        if max_batches is not None and batches >= max_batches:
            print()
            print("STOP")
            print("checked:", f"{checked:,}")
            print("best:", global_best_score)
            print("best_nonce:", global_best_nonce)
            print("best_hash:", global_best_hash)
            return global_best_score, global_best_nonce, global_best_hash

        if next_nonce > UINT64_MAX - batch_size:
            print("nonce空間を探索しきりました")
            return global_best_score, global_best_nonce, global_best_hash

        mine_max_f_kernel[blocks, threads](
            d_prefix,
            len(prefix_bytes),
            np.uint64(next_nonce),
            d_block_best_scores,
            d_block_best_nonces,
            d_K,
        )

        # ここでGPU完了待ち + 結果取得
        scores = d_block_best_scores.copy_to_host()
        nonces = d_block_best_nonces.copy_to_host()

        batch_best_score = int(scores.max())
        candidate_indices = np.flatnonzero(scores == batch_best_score)
        batch_best_nonce = int(nonces[candidate_indices].min())

        checked += batch_size
        next_nonce += batch_size
        batches += 1

        if batch_best_score > global_best_score:
            global_best_score = batch_best_score
            global_best_nonce = batch_best_nonce
            global_best_hash = sha256_hex(prefix_bytes, global_best_nonce)

            elapsed = time.perf_counter() - t0
            speed = checked / elapsed

            print(
                f"NEW BEST! "
                f"f={global_best_score} "
                f"nonce={global_best_nonce} "
                f"hash={global_best_hash} "
                f"checked={checked:,} "
                f"speed={speed:,.0f} H/s"
            )

        now = time.perf_counter()

        if now - last_progress >= progress_sec:
            elapsed = now - t0
            speed = checked / elapsed

            print(
                f"progress "
                f"checked={checked:,} "
                f"next_nonce={next_nonce:,} "
                f"speed={speed:,.0f} H/s "
                f"current_best_f={global_best_score} "
                f"best_nonce={global_best_nonce} "
                f"best_hash={global_best_hash}"
            )

            last_progress = now

In [ ]:
run(
    prefix="mumumu-",
    start=1453544439808,
    blocks=131072,
    threads=512,
    progress_sec=10
)

device: NVIDIA RTX PRO 6000 Blackwell Server Edition
prefix: b'mumumu-'
threads: 512
blocks: 131072
batch size: 67108864

compiling kernel...
start

NEW BEST! f=6 nonce=1453573467362 hash=ffffff450f0ad679ea41b20242fcbeecde3209b5f154b416fed62dbf0092a07d checked=67,108,864 speed=4,145,532,843 H/s


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


NEW BEST! f=7 nonce=1455340001381 hash=fffffff11e3d0e05aab4cfcd0d6771777d70d4ed1fde3911756b35851d3d37c2 checked=1,811,939,328 speed=4,207,342,574 H/s
NEW BEST! f=8 nonce=1462846259137 hash=ffffffffec206aaf9d39ea3ada441b7d586b80fc45b82c8813a4b027b9abf1bb checked=9,328,132,096 speed=4,216,557,800 H/s
NEW BEST! f=9 nonce=1493362830570 hash=fffffffff727b90130ad9957ce4391e2d71dfc38fb90124d734e03d245eb8cb7 checked=39,862,665,216 speed=4,217,013,721 H/s
progress checked=42,211,475,456 next_nonce=1,495,755,915,264 speed=4,216,946,683 H/s current_best_f=9 best_nonce=1493362830570 best_hash=fffffffff727b90130ad9957ce4391e2d71dfc38fb90124d734e03d245eb8cb7
progress checked=84,422,950,912 next_nonce=1,537,967,390,720 speed=4,217,689,882 H/s current_best_f=9 best_nonce=1493362830570 best_hash=fffffffff727b90130ad9957ce4391e2d71dfc38fb90124d734e03d245eb8cb7
progress checked=126,634,426,368 next_nonce=1,580,178,866,176 speed=4,217,680,198 H/s current_best_f=9 best_nonce=1493362830570 best_hash=fffffff

progress checked=3,871,577,473,024 next_nonce=5,325,121,912,832 speed=4,204,560,793 H/s current_best_f=10 best_nonce=4320734276682 best_hash=ffffffffffb7442fe727a55f68d565372278c8e2f7abe25610f74be444bd5d40
progress checked=3,913,654,730,752 next_nonce=5,367,199,170,560 speed=4,204,550,749 H/s current_best_f=10 best_nonce=4320734276682 best_hash=ffffffffffb7442fe727a55f68d565372278c8e2f7abe25610f74be444bd5d40
progress checked=3,955,731,988,480 next_nonce=5,409,276,428,288 speed=4,204,549,059 H/s current_best_f=10 best_nonce=4320734276682 best_hash=ffffffffffb7442fe727a55f68d565372278c8e2f7abe25610f74be444bd5d40
progress checked=3,997,809,246,208 next_nonce=5,451,353,686,016 speed=4,204,556,633 H/s current_best_f=10 best_nonce=4320734276682 best_hash=ffffffffffb7442fe727a55f68d565372278c8e2f7abe25610f74be444bd5d40
progress checked=4,039,886,503,936 next_nonce=5,493,430,943,744 speed=4,204,560,111 H/s current_best_f=10 best_nonce=4320734276682 best_hash=ffffffffffb7442fe727a55f68d56537227

KeyboardInterrupt: 